<a href="https://colab.research.google.com/github/Ymin-2/ESAA/blob/main/ESAA_OB_WEEK05_1_review5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**주제**

도서 추천 알고리즘 AI 모델 개발

https://dacon.io/competitions/official/236093/overview/description

##**데이터**

- train.csv
  - ID: 샘플 고유 ID
  - User-ID: 유저 고유 ID
  - Book-ID: 도서 고유 ID
  - 유저 정보
    - Age: 나이
    - Location: 지역
  - 도서 정보
    - Book-Title: 도서명
    - Book-Author: 도서 저자
    - Year-Of-Publication: 도서 출판 년도(-1일 경우 결측 혹은 알 수 없음)
    - Publisher: 출판사
  - Book-Rating: 유저가 도서에 부여한 평점(0점~10점)
    - 단, 0점인 경우에는 유저가 해당 도서에 관심이 없고 관련이 없는 경우

- test.csv
  - ID: 샘플 고유 ID
  - User-ID: 유저 고유 ID
  - Book-ID: 도서 고유 ID
  - 유저 정보
    - Age: 나이
    - Location: 지역
  - 도서 정보
    - Book-Title: 도서명
    - Book-Author: 도서 저자
    - Year-Of-Publication: 도서 출판 년도(-1일 경우 결측 혹은 알 수 없음)
    - Publisher: 출판사

- sample_submission.csv
   - ID: 샘플 고유 ID
   - Book-Rating: 유저가 도서에 부여한 평점(0점~10점)
    - 단, 0점인 경우에는 유저가 해당 도서에 관심이 없고 관련이 없는 경우

##**코드리뷰**

1. seed 고정
2. 문자열 전처리
  - Book-Title 정리
  - Main_Title, Sub_Title 분리
  - Location에서 City, State, Country 분리
3. 결측/이상치 처리
  - State, Country를 train 기준 최빈값으로 보정
  - Age 이상치 제거 후 평균 대체
  - Age_gb 구간화
4. 범주형 변수 인코딩
  - OrdinalEncoder(handle_unkown='use_encoded_value', unkown_vlaue=-2)
5. CatBoostRegressor 학습
  - Pool(,,, cat_features = FEATURE) 사용
6. 20-fold StratifiedKFold 검증
7. fold별 test 예측 평균
8. 예측값 clipping(0~10)

##**배울점**

- 텍스트형 컬럼은 바로 쓰지 않고 먼저 정리한 뒤 분해해서 사용
- 이상치 처리 후 구간화해서 범주형 정보로 바꿈
- CatBoost에서 Pool과 cat_feautres 명시
- clipping 적용

In [ ]:
# 텍스트 컬럼 분해
df['Book-Title'] = [re.sub(r'[^0-9a-zA-Z:,]',  ' ',str(i)) for i in df['Book-Title']]
df['Main_Title'] = [i.split('  ')[0] for i in df['Book-Title']]
df['Sub_Title'] = [''.join(i.split('  ')[1:]) for i in df['Book-Title']]
df['Sub_Title'] = np.where(df['Sub_Title'] == '', 'No_SUB', df['Sub_Title'])

In [ ]:
# 이상치 처리 후 구간화해서 범주형 정보로 변환
df.loc[(df['Age'] > 90) | (df['Age'] < 3), 'Age'] = np.nan

df['Age'] = df['Age'].fillna(train_lb['Age'].mean())
df['Age'] = df['Age'].astype(np.int32)

df['Age_gb'] = pd.cut(df.Age, bins, labels = labels, include_lowest = True)

In [ ]:
# CatBoost
FEATURE = ['User-ID', 'Main_Title','Sub_Title','Book-Author','Publisher', 'City','State','Country','Age_gb']

X_train_fold[FEATURE] = X_train_fold[FEATURE].astype('int')
X_valid_fold[FEATURE] = X_valid_fold[FEATURE].astype('int')

train_pool = Pool(data=X_train_fold, label=y_train_fold, cat_features=FEATURE)
valid_pool = Pool(data=X_valid_fold, label=y_valid_fold, cat_features=FEATURE)

In [ ]:
# clipping
y_pred = np.clip(model.predict(X_test), 0, 10)

y_test_pred += np.clip(fit_model.predict(x_test[X_valid_fold.columns]), 0.0, 10.0)